In [1]:
# Standard libraries
import os
import random
import numpy as np
import pandas as pd

# For preprocessing
import re
import string

# For tokenization & embeddings
from sentence_transformers import SentenceTransformer

# For classification
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, accuracy_score, roc_auc_score

# For visualization
import matplotlib.pyplot as plt
import seaborn as sns

# Plot settings
sns.set(style='whitegrid')
%matplotlib inline

# Reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)


ModuleNotFoundError: No module named 'sentence_transformers'

In [ ]:
# Load CSV
df = pd.read_csv("Def_Pairs_Paraphrased.csv")

# Show a few examples
print("Sample rows:")
display(df.head())

# Check label distribution
print("\nLabel distribution:")
print(df["term_lang"].value_counts())


In [ ]:
# Load SBERT model
sbert_model = SentenceTransformer("bert-base-nli-mean-tokens") 

# Combine conv + slang definitions into one string
combined_texts = df["conv_def_paraphrased"] + " [SEP] " + df["slang_def_paraphrased"]

# Compute embeddings
print("Encoding definitions...")
X = sbert_model.encode(combined_texts.tolist(), batch_size=32, show_progress_bar=True)

# Convert labels to binary: en -> 0, zh -> 1, ru -> 2
label_map = {"EN": 0, "ZH": 1, "RU":2}
y = df["term_lang"].map(label_map).values


In [ ]:
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix, ConfusionMatrixDisplay

def run_classifier(X, y, labels, test_size=0.1):
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=test_size, random_state=42, stratify=y
    )
    clf = LogisticRegression(max_iter=2500)
    clf.fit(X_train, y_train)
    y_pred = clf.predict(X_test)
    
    print(f"Labels: {labels}")
    print(f"Accuracy: {accuracy_score(y_test, y_pred):.4f}")
    print(f"F1 Score: {f1_score(y_test, y_pred, average='weighted'):.4f}")
    
    cm = confusion_matrix(y_test, y_pred)
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=labels)
    disp.plot(cmap="Blues")

run_classifier(X, y, labels=["EN", "ZH", "RU"])

pairs = [(0, 1), (1, 2), (0, 2)]
for a, b in pairs:
    mask = np.isin(y, [a, b])
    run_classifier(X[mask], y[mask], labels=[list(label_map.keys())[a], list(label_map.keys())[b]])
